# 第3章 债券定价原理 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch03_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch03_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 6：复现例3.1/3.2 + 价格—收益率曲线


In [ ]:
import numpy as np
from fi.cashflow import make_cashflows
from fi.pricing import price_bond
from fi import plotting
plotting.use_chinese_style()
cfs, ts = make_cashflows(0.03, 3, freq=1, face=100)
for y in (0.02, 0.03, 0.04):
    print(f'例3.1 y={y:.0%}: P={price_bond(cfs, ts, y, 1):.4f}')
print('例3.2 1Y零息@2.5% =', round(price_bond([100],[1],0.025,1),4), ' 3Y零息@2.5% =', round(price_bond([100],[3],0.025,1),4))
ys = np.linspace(0, 0.08, 161)
fig, ax = plotting.new_axes()
ax.plot(ys*100, [price_bond(cfs,ts,yi,1) for yi in ys])
ax.axhline(100, ls=':', color='gray'); ax.axvline(3, ls=':', color='gray')
ax.set_xlabel('到期收益率 (%)'); ax.set_ylabel('价格'); ax.set_title('价格—收益率反向变动')
fig.tight_layout()


## 编程实验 7：拉回面值（溢价债 vs 折价债）


In [ ]:
mats = np.arange(10, 0-1e-9, -1)
def price_at(coupon, y, mat):
    if mat <= 0: return 100.0
    cf, t = make_cashflows(coupon, mat, 1, 100); return price_bond(cf, t, y, 1)
prem = [price_at(0.04, 0.025, m) for m in mats]   # 票息4% > 收益率2.5% 溢价
disc = [price_at(0.015, 0.025, m) for m in mats]  # 票息1.5% < 收益率2.5% 折价
fig, ax = plotting.new_axes()
ax.plot(mats, prem, label='溢价债'); ax.plot(mats, disc, label='折价债')
ax.axhline(100, ls=':', color='gray'); ax.invert_xaxis()
ax.set_xlabel('剩余期限（年）'); ax.set_ylabel('价格'); ax.set_title('拉回面值：到期临近收敛到 100'); ax.legend()
fig.tight_layout()


## 编程实验 8：fi vs QuantLib 对拍误差表


In [ ]:
import QuantLib as ql
today = ql.Date(15,6,2026); ql.Settings.instance().evaluationDate = today
sched = ql.Schedule(today, today+ql.Period(3,ql.Years), ql.Period(ql.Annual),
                    ql.NullCalendar(), ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Backward, False)
dc = ql.ActualActual(ql.ActualActual.ISDA)
bond = ql.FixedRateBond(0, 100.0, sched, [0.03], dc)
print(f"{'y':>5}{'fi':>12}{'QuantLib':>12}{'误差':>11}")
for y in (0.02, 0.03, 0.04):
    p_fi = price_bond(cfs, ts, y, 1)
    p_ql = ql.BondFunctions.cleanPrice(bond, ql.InterestRate(y, dc, ql.Compounded, ql.Annual))
    print(f'{y:>5.0%}{p_fi:>12.4f}{p_ql:>12.4f}{abs(p_fi-p_ql):>11.2e}')
print('差异来源：计息惯例(ActualActual)/付息频率/结算日；整年无调整时误差在 1e-6 量级')
